# DL400 Exercises -- RNNs, LSTMs, and GRUs

Work through these exercises after completing the DL400 module. Try each exercise before looking at the solutions at the bottom.

**Topics covered:** RNN hidden state mechanics, LSTM gate equations, GRU architecture, embedding dimensions, handling variable-length sequences.

---
## Exercise 1: RNN Hidden State Step-by-Step

A vanilla RNN cell computes:

$$h_t = \tanh(W_{ih} x_t + b_{ih} + W_{hh} h_{t-1} + b_{hh})$$

Given:
- Input dimension: 3, Hidden dimension: 2
- Weights (simplified): $W_{ih} = \begin{bmatrix} 0.5 & 0.3 & -0.2 \\ 0.1 & -0.4 & 0.6 \end{bmatrix}$, $W_{hh} = \begin{bmatrix} 0.2 & -0.1 \\ 0.3 & 0.4 \end{bmatrix}$
- Biases: all zeros
- Initial hidden state: $h_0 = [0, 0]$
- Input sequence: $x_1 = [1, 0, -1]$, $x_2 = [0, 1, 0.5]$

1. Compute $h_1$ (hidden state after processing $x_1$)
2. Compute $h_2$ (hidden state after processing $x_2$)
3. Verify your answers with PyTorch

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# Manual computation
W_ih = np.array([[0.5, 0.3, -0.2],
                 [0.1, -0.4, 0.6]])
W_hh = np.array([[0.2, -0.1],
                 [0.3, 0.4]])

h0 = np.array([0.0, 0.0])
x1 = np.array([1.0, 0.0, -1.0])
x2 = np.array([0.0, 1.0, 0.5])

# Step 1: h1 = tanh(W_ih @ x1 + W_hh @ h0)
# TODO: compute h1

# Step 2: h2 = tanh(W_ih @ x2 + W_hh @ h1)
# TODO: compute h2

# Verify with PyTorch
rnn = nn.RNN(input_size=3, hidden_size=2, batch_first=True, bias=False)
# TODO: Set weights manually and verify

---
## Exercise 2: LSTM Gate Analysis

An LSTM cell has four components:

- **Forget gate:** $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$
- **Input gate:** $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$
- **Cell candidate:** $\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)$
- **Output gate:** $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$
- **Cell state:** $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$
- **Hidden state:** $h_t = o_t \odot \tanh(c_t)$

Answer the following conceptual questions:

1. If the forget gate outputs all 0s ($f_t = \mathbf{0}$), what happens to the cell state? What does this mean conceptually?
2. If the forget gate outputs all 1s ($f_t = \mathbf{1}$) and input gate outputs all 0s ($i_t = \mathbf{0}$), what happens?
3. Why does the LSTM use the cell state $c_t$ (a separate "memory highway") instead of just the hidden state? How does this help with vanishing gradients?
4. What activation function is used for gates and why? What would happen if we used ReLU instead?

**Your answers:**

---
## Exercise 3: GRU vs LSTM Comparison

A GRU (Gated Recurrent Unit) has:

- **Reset gate:** $r_t = \sigma(W_r [h_{t-1}, x_t])$
- **Update gate:** $z_t = \sigma(W_z [h_{t-1}, x_t])$
- **Candidate:** $\tilde{h}_t = \tanh(W [r_t \odot h_{t-1}, x_t])$
- **Hidden state:** $h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$

1. How many gates does GRU have vs LSTM? What is the parameter count difference?
2. GRU has no separate cell state. How does the update gate $z_t$ combine the roles of the LSTM forget and input gates?
3. Create both an LSTM and GRU in PyTorch with the same input/hidden dimensions and compare parameter counts.

In [ ]:
import torch.nn as nn

input_size = 64
hidden_size = 128

lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
gru = nn.GRU(input_size, hidden_size, batch_first=True)

lstm_params = sum(p.numel() for p in lstm.parameters())
gru_params = sum(p.numel() for p in gru.parameters())

print(f"LSTM parameters: {lstm_params:,}")
print(f"GRU parameters:  {gru_params:,}")
print(f"GRU is {gru_params/lstm_params:.1%} the size of LSTM")

# TODO: Explain the parameter count formula for each
# LSTM: 4 * (input_size * hidden_size + hidden_size * hidden_size + hidden_size + hidden_size)
# GRU:  3 * (input_size * hidden_size + hidden_size * hidden_size + hidden_size + hidden_size)

---
## Exercise 4: Embedding Dimensions

Answer the following about word embeddings:

1. You have a vocabulary of 50,000 words and want to use 300-dimensional embeddings. How many parameters does the embedding layer have?

2. If you set `padding_idx=0`, what happens to the embedding vector for index 0? Why is this useful?

3. Your model takes integer token IDs as input. What happens if you feed it an ID that is >= vocab_size? Why?

4. You are training a text classifier. Should embedding dimensions be: (a) Equal to vocab size, (b) Much smaller than vocab size, or (c) Much larger than vocab size? Explain.

In [ ]:
import torch
import torch.nn as nn

# Verify parameter count
vocab_size = 50000
embed_dim = 300

embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
n_params = sum(p.numel() for p in embedding.parameters())
print(f"Embedding parameters: {n_params:,}")

# Check padding_idx behavior
print(f"\nEmbedding for PAD (index 0): {embedding(torch.tensor([0]))}")
print(f"All zeros? {(embedding(torch.tensor([0])) == 0).all()}")

# What happens with out-of-range indices?
try:
    embedding(torch.tensor([50001]))
except IndexError as e:
    print(f"\nOut-of-range error: {e}")

---
## Exercise 5: Variable-Length Sequences

In NLP, sentences have different lengths. You need to handle this with padding and masking.

Given three sentences tokenized as:
- Sentence 1: [5, 12, 3, 7, 9]
- Sentence 2: [8, 2, 6]
- Sentence 3: [1, 4, 11, 10]

1. Pad all sequences to the maximum length using PAD token (index 0)
2. Create an attention mask (1 for real tokens, 0 for padding)
3. Pass through an LSTM and extract the **last real hidden state** for each sequence (not the last padded position)

In [ ]:
import torch
import torch.nn as nn

# Raw sequences (different lengths)
sequences = [
    [5, 12, 3, 7, 9],
    [8, 2, 6],
    [1, 4, 11, 10],
]

# TODO:
# 1. Pad to max length with 0
# 2. Create mask tensor
# 3. Get true lengths

# padded = ?  # shape: (3, 5)
# mask = ?    # shape: (3, 5)
# lengths = ?  # [5, 3, 4]

# 4. Pass through embedding + LSTM
# embedding = nn.Embedding(20, 32, padding_idx=0)
# lstm = nn.LSTM(32, 64, batch_first=True)

# 5. Extract the hidden state at the last REAL position for each sequence
#    (NOT the hidden state at position 4 for all sequences)

---
## Exercise 6: Bidirectional RNN Output Shapes

When using `bidirectional=True` in PyTorch's LSTM/GRU:

1. If `hidden_size=128` and `bidirectional=True`, what is the output size at each timestep?
2. What is the shape of the hidden state `h_n` returned by the LSTM?
3. For classification, how do you combine the forward and backward final hidden states?

Verify your answers with code.

In [ ]:
import torch
import torch.nn as nn

batch_size = 4
seq_len = 10
input_size = 64
hidden_size = 128
num_layers = 2

lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
               batch_first=True, bidirectional=True)

x = torch.randn(batch_size, seq_len, input_size)
output, (h_n, c_n) = lstm(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")  # TODO: predict this
print(f"h_n shape:    {h_n.shape}")     # TODO: predict this
print(f"c_n shape:    {c_n.shape}")     # TODO: predict this

# TODO: Extract final forward and backward hidden states from the LAST layer
# h_forward = ?
# h_backward = ?
# h_combined = torch.cat([h_forward, h_backward], dim=1)  # for classification

---
## Exercise 7: Sequence Classification Architecture

Design and implement a text classification model using an LSTM. The model should:

1. Use an embedding layer
2. Use a bidirectional LSTM with 2 layers
3. Apply dropout between LSTM layers
4. Use the concatenated final hidden states for classification
5. Have a classification head with one hidden layer

Test with a random batch and verify all shapes.

In [ ]:
import torch
import torch.nn as nn

class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        # TODO: Define layers
        # - Embedding with padding_idx
        # - Bidirectional LSTM with num_layers and dropout
        # - Classification head: Linear -> ReLU -> Dropout -> Linear
        pass

    def forward(self, x):
        # TODO:
        # 1. Embed tokens
        # 2. Pass through LSTM
        # 3. Extract final hidden states (forward + backward)
        # 4. Concatenate and pass through classifier
        pass


# Test
# model = TextClassifier(vocab_size=5000, embed_dim=128, hidden_dim=64,
#                        num_classes=4, num_layers=2, dropout=0.3)
# x = torch.randint(0, 5000, (8, 50))  # (batch=8, seq_len=50)
# output = model(x)
# print(f"Output shape: {output.shape}")  # Should be (8, 4)

---
---
# Solutions

Expand each section below to see the solution.

<details>
<summary><b>Solution 1: RNN Hidden State Step-by-Step</b></summary>

```python
import numpy as np

W_ih = np.array([[0.5, 0.3, -0.2],
                 [0.1, -0.4, 0.6]])
W_hh = np.array([[0.2, -0.1],
                 [0.3, 0.4]])
h0 = np.array([0.0, 0.0])
x1 = np.array([1.0, 0.0, -1.0])
x2 = np.array([0.0, 1.0, 0.5])

# Step 1: h1
z1 = W_ih @ x1 + W_hh @ h0
# z1 = [0.5*1 + 0.3*0 + (-0.2)*(-1), 0.1*1 + (-0.4)*0 + 0.6*(-1)] + [0, 0]
# z1 = [0.7, -0.5]
h1 = np.tanh(z1)  # [tanh(0.7), tanh(-0.5)] = [0.6044, -0.4621]
print(f"h1 = {h1}")

# Step 2: h2
z2 = W_ih @ x2 + W_hh @ h1
# W_ih @ x2 = [0.3 + (-0.1), -0.4 + 0.3] = [0.2, -0.1]
# W_hh @ h1 = [0.2*0.6044 + (-0.1)*(-0.4621), 0.3*0.6044 + 0.4*(-0.4621)]
#           = [0.1671, -0.0035]
# z2 = [0.2 + 0.1671, -0.1 + (-0.0035)] = [0.3671, -0.1035]
h2 = np.tanh(z2)
print(f"h2 = {h2}")
```

To verify with PyTorch, set the RNN weights manually using `rnn.weight_ih_l0.data` and `rnn.weight_hh_l0.data`.
</details>

<details>
<summary><b>Solution 2: LSTM Gate Analysis</b></summary>

1. **Forget gate = all 0s:** The cell state becomes $c_t = 0 \cdot c_{t-1} + i_t \cdot \tilde{c}_t$. The previous cell state is completely erased. Conceptually, the LSTM "forgets everything" it previously remembered and starts fresh. This is useful when encountering a topic change or sentence boundary.

2. **Forget gate = all 1s, input gate = all 0s:** The cell state becomes $c_t = 1 \cdot c_{t-1} + 0 \cdot \tilde{c}_t = c_{t-1}$. The cell state is perfectly preserved -- no new information is written, and nothing is forgotten. This allows the LSTM to carry information unchanged across many timesteps.

3. **Separate cell state for vanishing gradients:** The cell state $c_t$ uses *additive* updates (multiplied by forget gate, then added to). During backpropagation, the gradient flows through the cell state via multiplication by the forget gate values (which are between 0 and 1). When the forget gate is close to 1, the gradient flows almost unchanged, creating a "gradient highway" that prevents vanishing gradients over long sequences. A vanilla RNN applies tanh at every step, which repeatedly squashes the gradient.

4. **Sigmoid for gates:** The sigmoid function outputs values in (0, 1), which naturally models the concept of "how much to let through" (a gate). If we used ReLU (output in [0, inf)), the gates could amplify signals unboundedly, leading to exploding values. The bounded output of sigmoid provides stable gating behavior.
</details>

<details>
<summary><b>Solution 3: GRU vs LSTM Comparison</b></summary>

1. **Gate count:** GRU has 2 gates (reset, update). LSTM has 3 gates + 1 candidate (forget, input, output, cell candidate). GRU has 75% of LSTM's parameters.

2. **GRU's update gate combines forget and input:** The update gate $z_t$ controls both forgetting and remembering simultaneously:
   - $h_t = (1 - z_t) \cdot h_{t-1} + z_t \cdot \tilde{h}_t$
   - When $z_t \approx 0$: keep old state (like LSTM forget=1, input=0)
   - When $z_t \approx 1$: replace with new candidate (like LSTM forget=0, input=1)
   - This coupling means GRU cannot independently control forgetting and remembering, but in practice it works well with fewer parameters.

3. **Parameter count:**
   ```
   For input_size=64, hidden_size=128:
   LSTM: 4 * (64*128 + 128*128 + 128 + 128) = 4 * 24,832 = 99,328
   GRU:  3 * (64*128 + 128*128 + 128 + 128) = 3 * 24,832 = 74,496
   GRU is 75% the size of LSTM.
   ```
</details>

<details>
<summary><b>Solution 4: Embedding Dimensions</b></summary>

1. **Parameters:** 50,000 x 300 = **15,000,000** (15M parameters). Each word gets a 300-dimensional vector.

2. **padding_idx=0:** The embedding vector for index 0 is always a zero vector and its gradient is always zero (never updated during training). This is useful because padding tokens should not carry information -- a zero vector ensures padding contributes nothing when processed by subsequent layers.

3. **Out-of-range index:** PyTorch raises an `IndexError`. The embedding layer is essentially a lookup table of size `(vocab_size, embed_dim)`. Requesting index >= vocab_size is like accessing an array out of bounds.

4. **Embedding dimension choice: (b) Much smaller than vocab size.** 
   - Typical values: 50-300 for small tasks, 768-1024 for large models.
   - (a) Equal to vocab_size (e.g., 50,000-dim) is wasteful -- most of that space is unused because words live on a much lower-dimensional manifold.
   - (c) Larger than vocab_size makes no sense -- you'd have more dimensions than words.
   - The embedding layer learns to map each word into a dense, low-dimensional space where semantic relationships are captured by distance and direction.
</details>

<details>
<summary><b>Solution 5: Variable-Length Sequences</b></summary>

```python
import torch
import torch.nn as nn

sequences = [
    [5, 12, 3, 7, 9],
    [8, 2, 6],
    [1, 4, 11, 10],
]

# 1. Pad to max length
max_len = max(len(s) for s in sequences)
padded = [s + [0] * (max_len - len(s)) for s in sequences]
padded = torch.tensor(padded, dtype=torch.long)
print(f"Padded:\n{padded}")
# [[5, 12, 3, 7, 9],
#  [8, 2, 6, 0, 0],
#  [1, 4, 11, 10, 0]]

# 2. Create mask
mask = (padded != 0).long()
print(f"Mask:\n{mask}")

# 3. True lengths
lengths = [len(s) for s in sequences]  # [5, 3, 4]

# 4. Embedding + LSTM
embedding = nn.Embedding(20, 32, padding_idx=0)
lstm = nn.LSTM(32, 64, batch_first=True)

embedded = embedding(padded)  # (3, 5, 32)
output, (h_n, c_n) = lstm(embedded)  # output: (3, 5, 64)

# 5. Extract hidden state at the LAST REAL position
last_hidden = []
for i, length in enumerate(lengths):
    last_hidden.append(output[i, length - 1, :])  # index into the real last position
last_hidden = torch.stack(last_hidden)  # (3, 64)
print(f"Last real hidden states shape: {last_hidden.shape}")

# Alternative using pack_padded_sequence (more efficient):
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
packed = pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=False)
packed_output, (h_n_packed, _) = lstm(packed)
# h_n_packed[-1] already contains the last REAL hidden state for each sequence
```
</details>

<details>
<summary><b>Solution 6: Bidirectional RNN Output Shapes</b></summary>

```python
import torch
import torch.nn as nn

batch_size, seq_len, input_size = 4, 10, 64
hidden_size, num_layers = 128, 2

lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
               batch_first=True, bidirectional=True)
x = torch.randn(batch_size, seq_len, input_size)
output, (h_n, c_n) = lstm(x)

# output: (batch, seq_len, hidden_size * 2) = (4, 10, 256)
# Each timestep has both forward and backward hidden states concatenated.

# h_n: (num_layers * 2, batch, hidden_size) = (4, 4, 128)
# Layout: [layer0_forward, layer0_backward, layer1_forward, layer1_backward]

# Extract final hidden states from the LAST layer:
h_forward = h_n[-2]   # last layer, forward direction:  (batch, 128)
h_backward = h_n[-1]  # last layer, backward direction: (batch, 128)
h_combined = torch.cat([h_forward, h_backward], dim=1)  # (batch, 256)

print(f"Output: {output.shape}")       # (4, 10, 256)
print(f"h_n: {h_n.shape}")             # (4, 4, 128)
print(f"h_combined: {h_combined.shape}") # (4, 256)
```

**Key insight:** For bidirectional LSTMs with `num_layers=L`, `h_n` has shape `(2*L, batch, hidden)`. The last two entries (`h_n[-2]` and `h_n[-1]`) are the forward and backward final hidden states from the last layer.
</details>

<details>
<summary><b>Solution 7: Sequence Classification Architecture</b></summary>

```python
import torch
import torch.nn as nn

class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        # hidden_dim * 2 because bidirectional
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))  # (batch, seq, embed)
        output, (h_n, c_n) = self.lstm(embedded)
        # Concatenate forward and backward final hidden states
        h_cat = torch.cat([h_n[-2], h_n[-1]], dim=1)  # (batch, hidden*2)
        h_cat = self.dropout(h_cat)
        logits = self.classifier(h_cat)
        return logits

model = TextClassifier(
    vocab_size=5000, embed_dim=128, hidden_dim=64,
    num_classes=4, num_layers=2, dropout=0.3
)
x = torch.randint(0, 5000, (8, 50))
output = model(x)
print(f"Output shape: {output.shape}")  # (8, 4)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
```
</details>